In [1]:
import re
import json
from pathlib import Path
from typing import List, Dict, Any

In [2]:
# Configuración de rutas
# INPUT_FILE = "../../outputs/processed/shpNTpo_txt/all.txt"
INPUT_FILE = "../../outputs/processed/spavbl_txt/all.txt"
OUTPUT_FILE = "spa_bible_corpus.json"

In [3]:
BOOKS_BIBLE = {
    "GEN": "GÉNESIS",
    "RUT": "RUT",
    "JOL": "JOEL",
    "JON": "JONÁS",
    "HEB": "HABACUC",
    "MAL": "MALAQUÍAS",
    "MAT": "MATEO",
    "MRK": "SAN MARCOS",
    "LUK": "SAN LUCAS",
    "1JN": "JUAN",
    "ACT": "HECHOS",
    "ROM": "ROMANOS",
    "1CO": "1 CORINTIOS",
    "2CO": "2 CORINTIOS",
    "GAL": "GÁLATAS",
    "EPH": "EFESIOS",
    "PHP": "FILIPENSES",
    "COL": "COLOSENSES",
    "1TH": "1 TESALONICENSES",
    "2TH": "2 TESALONICENSES",
    "1TI": "1 TIMOTEO",
    "2TI": "2 TIMOTEO",
    "TIT": "TITO",
    "PHM": "FILEMÓN",
    "HEB": "HEBREOS",
    "JAS": "SANTIAGO",
    "1PE": "1 PEDRO",
    "2PE": "2 PEDRO",
    "1JN": "1 JUAN",
    "2JN": "2 JUAN",
    "3JN": "3 JUAN",
    "JUD": "JUDAS",
    "REV": "APOCALIPSIS"
}

In [ ]:
## Funciones de procesamiento

def load_text_file(filepath: str) -> str:
    """Carga el archivo de texto con manejo de errores."""
    try:
        with open(filepath, "r", encoding="utf-8-sig") as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {filepath}")
        raise
    except Exception as e:
        print(f"Error al leer el archivo: {e}")
        raise

def clean_verse_text(text: str) -> str:
    """Limpia el texto del versículo eliminando espacios extras y caracteres no deseados."""
    # Eliminar símbolos de notas al pie
    text = re.sub(r'[*†‡§]', '', text)
    # Eliminar espacios múltiples
    text = re.sub(r'\s+', ' ', text)
    # Eliminar espacios al inicio y final
    text = text.strip()
    return text

def extract_book_content(content: str, book_key: str) -> str:
    """
    Extrae el contenido completo de un libro específico del archivo.
    """
    # Buscar el inicio del libro actual
    start_pattern = rf"=+\s*{re.escape(book_key)}\.htm\s*=+"
    start_match = re.search(start_pattern, content)
    
    if not start_match:
        return ""
    
    start_pos = start_match.start()
    
    # Buscar el inicio del siguiente libro (cualquier otro código de 3 letras)
    next_book_pattern = r"=+\s*[A-Z0-9]{3}\.htm\s*=+"
    remaining_content = content[start_match.end():]
    next_match = re.search(next_book_pattern, remaining_content)
    
    if next_match:
        # Hay un siguiente libro, tomar solo hasta ahí
        end_pos = start_match.end() + next_match.start()
        book_content = content[start_pos:end_pos]
    else:
        # Es el último libro, tomar hasta el final
        book_content = content[start_pos:]
    
    return book_content

def extract_chapters(book_content: str, book_key: str, book_name: str) -> List[Dict[str, Any]]:
    """
    Extrae todos los capítulos de un libro.
    """
    chapters = []
    
    # Patrón para dividir por capítulos: ===== BOOK##.htm =====
    chapter_pattern = rf"=+\s*{re.escape(book_key)}(\d{{2}})\.htm\s*=+"
    
    # Encontrar todas las posiciones de capítulos
    chapter_matches = list(re.finditer(chapter_pattern, book_content))
    
    for i, match in enumerate(chapter_matches):
        chapter_num = int(match.group(1))
        
        # Determinar el inicio y fin del contenido del capítulo
        start_pos = match.end()
        
        if i + 1 < len(chapter_matches):
            # Hay un siguiente capítulo
            end_pos = chapter_matches[i + 1].start()
        else:
            # Es el último capítulo
            end_pos = len(book_content)
        
        chapter_content = book_content[start_pos:end_pos]
        
        # Extraer versículos de este capítulo
        verses = extract_verses_from_chapter(chapter_content, book_name, chapter_num)
        
        if verses:
            chapters.append({
                "chapter": chapter_num,
                "verse_count": len(verses),
                "verses": verses
            })
    
    return chapters

def extract_verses_from_chapter(chapter_content: str, book_name: str, chapter_num: int) -> List[Dict[str, Any]]:
    """
    Extrae los versículos de un capítulo específico.
    """
    verses = []
    
    # Dividir en líneas
    lines = chapter_content.split('\n')
    
    current_verse_num = None
    current_verse_text = []
    
    for line in lines:
        line = line.strip()
        
        # Saltar líneas vacías, encabezados, navegación, etc.
        if not line:
            continue
        if line.startswith('•'):
            continue
        if line == book_name:
            continue
        if f"{book_name} {chapter_num}" in line:
            continue
        if line.startswith('<') or line.startswith('>'):
            continue
        if line.startswith('Versión Biblia Libre'):
            continue
        if line.startswith('©'):
            break  # Llegamos al copyright, fin del contenido
        
        # Buscar inicio de versículo: número seguido de espacio y texto
        verse_match = re.match(r'^(\d+)\s+(.+)', line)
        
        if verse_match:
            # Guardar el versículo anterior si existe
            if current_verse_num is not None and current_verse_text:
                verse_text = ' '.join(current_verse_text)
                verse_text = clean_verse_text(verse_text)
                if verse_text:
                    verses.append({
                        "verse": current_verse_num,
                        "text": verse_text
                    })
            
            # Iniciar nuevo versículo
            current_verse_num = int(verse_match.group(1))
            current_verse_text = [verse_match.group(2)]
        
        elif current_verse_num is not None:
            # Continuar texto del versículo actual
            # Verificar que no sea una nota al pie o metadatos
            if not line.startswith('*') and not line.startswith('†') and not line.startswith('‡'):
                current_verse_text.append(line)
    
    # Guardar el último versículo
    if current_verse_num is not None and current_verse_text:
        verse_text = ' '.join(current_verse_text)
        verse_text = clean_verse_text(verse_text)
        if verse_text:
            verses.append({
                "verse": current_verse_num,
                "text": verse_text
            })
    
    return verses

def process_book(content: str, book_key: str, book_name: str) -> Dict[str, Any]:
    """
    Procesa un libro completo de la Biblia.
    """
    # Extraer el contenido de este libro específico
    book_content = extract_book_content(content, book_key)
    
    if not book_content:
        return {
            "book_key": book_key,
            "book_name": book_name,
            "chapter_count": 0,
            "chapters": []
        }
    
    # Extraer capítulos
    chapters = extract_chapters(book_content, book_key, book_name)
    
    return {
        "book_key": book_key,
        "book_name": book_name,
        "chapter_count": len(chapters),
        "chapters": chapters
    }

def build_corpus(content: str, books: Dict[str, str]) -> List[Dict[str, Any]]:
    """Construye el corpus completo de la Biblia."""
    corpus = []
    
    print("Procesando libros...\n")
    
    for book_key, book_name in books.items():
        print(f"Procesando {book_name} ({book_key})...", end=" ")
        
        book_data = process_book(content, book_key, book_name)
        
        # Solo agregar libros que tengan capítulos
        if book_data["chapters"]:
            total_verses = sum(ch["verse_count"] for ch in book_data["chapters"])
            corpus.append(book_data)
            print(f"✓ {book_data['chapter_count']} capítulos, {total_verses} versículos")
        else:
            print(f"⚠ No se encontró contenido")
    
    return corpus

def save_corpus(corpus: List[Dict[str, Any]], filepath: str) -> None:
    """Guarda el corpus en formato JSON."""
    try:
        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(corpus, f, ensure_ascii=False, indent=2)
        print(f"\n✓ Corpus guardado en: {filepath}")
    except Exception as e:
        print(f"Error al guardar el archivo: {e}")
        raise

In [ ]:
## Funciones de visualización

def print_corpus_summary(corpus: List[Dict[str, Any]]) -> None:
    """Imprime un resumen del corpus generado."""
    print("\n" + "="*60)
    print("RESUMEN DEL CORPUS")
    print("="*60)
    
    total_chapters = 0
    total_verses = 0
    
    for book in corpus:
        book_verses = sum(ch["verse_count"] for ch in book["chapters"])
        total_chapters += book["chapter_count"]
        total_verses += book_verses
        
        print(f"\n{book['book_name']} ({book['book_key']}):")
        print(f"  - Capítulos: {book['chapter_count']}")
        print(f"  - Versículos: {book_verses}")
    
    print(f"\n{'='*60}")
    print(f"TOTAL: {len(corpus)} libros, {total_chapters} capítulos, {total_verses} versículos")
    print("="*60)

def print_sample_verses(corpus: List[Dict[str, Any]], num_books: int = 3) -> None:
    """Muestra ejemplos de versículos de los primeros libros."""
    print("\n" + "="*60)
    print("EJEMPLOS DE VERSÍCULOS")
    print("="*60)
    
    for book in corpus[:num_books]:
        if book["chapters"]:
            first_chapter = book["chapters"][0]
            print(f"\n{book['book_name']} - Capítulo {first_chapter['chapter']}")
            print("-" * 60)
            
            # Mostrar los primeros 3 versículos
            for verse in first_chapter["verses"][:3]:
                print(f"{verse['verse']}. {verse['text']}")

In [ ]:
## Ejecución principal

def main():
    """Función principal."""
    print("="*60)
    print("PROCESAMIENTO DE CORPUS BÍBLICO")
    print("="*60)
    print(f"\nArchivo de entrada: {INPUT_FILE}")
    print(f"Archivo de salida: {OUTPUT_FILE}\n")
    
    # Cargar archivo
    print("Cargando archivo...", end=" ")
    content = load_text_file(INPUT_FILE)
    print(f"✓ {len(content)} caracteres\n")
    
    # Construir corpus
    corpus = build_corpus(content, BOOKS_BIBLE)
    
    # Mostrar resumen
    print_corpus_summary(corpus)
    
    # Guardar resultado
    save_corpus(corpus, OUTPUT_FILE)
    
    # Mostrar ejemplos
    print_sample_verses(corpus, num_books=3)
    
    return corpus

# Ejecutar el procesamiento
corpus = main()

# ## Funciones auxiliares de búsqueda

def search_verse(corpus: List[Dict], book_key: str, chapter: int, verse: int) -> str:
    """
    Busca un versículo específico en el corpus.
    """
    for book in corpus:
        if book['book_key'] == book_key:
            for ch in book['chapters']:
                if ch['chapter'] == chapter:
                    for v in ch['verses']:
                        if v['verse'] == verse:
                            return v['text']
    return "Versículo no encontrado"

def get_book_stats(corpus: List[Dict], book_key: str) -> Dict[str, Any]:
    """
    Obtiene estadísticas de un libro específico.
    """
    for book in corpus:
        if book['book_key'] == book_key:
            total_verses = sum(ch['verse_count'] for ch in book['chapters'])
            return {
                'book_name': book['book_name'],
                'chapters': book['chapter_count'],
                'verses': total_verses,
                'avg_verses_per_chapter': total_verses / book['chapter_count'] if book['chapter_count'] > 0 else 0
            }
    return None

PROCESAMIENTO DE CORPUS BÍBLICO

Archivo de entrada: ../../outputs/processed/spavbl_txt/all.txt
Archivo de salida: spa_bible_corpus.json

Cargando archivo... ✓ 5225179 caracteres

Procesando libros...

Procesando GÉNESIS (GEN)... ✓ 50 capítulos, 692 versículos
Procesando RUT (RUT)... ✓ 4 capítulos, 49 versículos
Procesando JOEL (JOL)... ✓ 3 capítulos, 20 versículos
Procesando JONÁS (JON)... ✓ 4 capítulos, 18 versículos
Procesando HEBREOS (HEB)... ✓ 13 capítulos, 77 versículos
Procesando MALAQUÍAS (MAL)... ✓ 4 capítulos, 21 versículos
Procesando MATEO (MAT)... ✓ 28 capítulos, 346 versículos
Procesando SAN MARCOS (MRK)... ✓ 16 capítulos, 291 versículos
Procesando SAN LUCAS (LUK)... ✓ 24 capítulos, 422 versículos
Procesando 1 JUAN (1JN)... ✓ 5 capítulos, 24 versículos
Procesando HECHOS (ACT)... ✓ 28 capítulos, 312 versículos
Procesando ROMANOS (ROM)... ✓ 16 capítulos, 80 versículos
Procesando 1 CORINTIOS (1CO)... ✓ 16 capítulos, 81 versículos
Procesando 2 CORINTIOS (2CO)... ✓ 13 capítulos

In [ ]:
# Ejemplo de búsqueda
if corpus:
    print("\n" + "="*60)
    print("CONSULTAR POR LIBRO")
    print("="*60)
    
    # Buscar un versículo específico del primer libro disponible
    if corpus[0]['chapters'] and corpus[0]['chapters'][0]['verses']:
        first_book = corpus[0]
        first_chapter = first_book['chapters'][0]
        first_verse = first_chapter['verses'][0]
        
        result = search_verse(corpus, first_book['book_key'], 
                            first_chapter['chapter'], first_verse['verse'])
        
        print(f"\n{first_book['book_name']} {first_chapter['chapter']}:{first_verse['verse']}")
        print(f"{result}")
        
    # Mostrar estadísticas del primer libro
    if corpus:
        stats = get_book_stats(corpus, corpus[0]['book_key'])
        if stats:
            print(f"\n\nEstadísticas de {stats['book_name']}:")
            print(f"  - Capítulos: {stats['chapters']}")
            print(f"  - Versículos totales: {stats['verses']}")
            print(f"  - Promedio de versículos por capítulo: {stats['avg_verses_per_chapter']:.1f}")


EJEMPLO DE BÚSQUEDA

GÉNESIS 1:1
En el principio, Dios creó los cielos y la tierra. 2 La tierra carecía de forma y estaba vacía; la oscuridad cubría la superficie del abismo y el Espíritu de Dios se movía sobre la superficie de las aguas.


Estadísticas de GÉNESIS:
  - Capítulos: 50
  - Versículos totales: 692
  - Promedio de versículos por capítulo: 13.8
